# DisasterLens on a Colab GPU

This notebook pulls the repository into the Colab runtime, installs it, and verifies the GPU. Run it with the VS Code Colab kernel selected.

In [ ]:
import os
import subprocess
from pathlib import Path

try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = os.environ.get("GITHUB_TOKEN")

if not github_token:
    raise RuntimeError("Add a GITHUB_TOKEN secret in Colab before running this cell.")

REPO_URL = os.environ.get("DISASTERLENS_REPO_URL", "https://github.com/kushc2004/disaster-lens.git")
REPO_DIR = Path("/content/disaster-lens")

git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader", "GIT_CONFIG_VALUE_0": f"Authorization: Bearer {github_token}"})

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], env=git_env, check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], env=git_env, check=True)

%cd /content/disaster-lens

In [ ]:
%pip install -e .

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Select a Colab GPU kernel before running this notebook."

In [ ]:
# Current repository smoke check
!python scripts/create_smoke_bright.py
!python scripts/inspect_bright.py data=bright dataset.root=data/samples/bright_smoke
!pytest